# 🎨 Exploration du Dataset CelebA & Analyse Complète

**Exploration complète du dataset CelebA** avec 202,599 images, 40 attributs, et 10,177 individus uniques.

Ce notebook:
1. Charge les métadonnées CelebA (attributs, identités, repères)
2. Analyse les distributions et les patterns
3. Crée 12+ graphiques de visualisation
4. Génère les statistiques pour l'entraînement

## Cellule 1: Configuration et Importations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# Configuration du style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print('✅ Importations réussies')

KeyboardInterrupt: 

## Cellule 2: Configuration des Chemins

In [ ]:
# Configuration des chemins
PROJECT_ROOT = Path('..') if Path('../DATASET').exists() else Path('.')
DATASET_ROOT = PROJECT_ROOT / 'DATASET'
METADATA_DIR = DATASET_ROOT / 'CelebA' / 'metadata'
IMG_DIR = DATASET_ROOT / 'CelebA' / 'img_align_celeba'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'data_exploration'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'✓ Chemins configurés. Dossier de sortie: {OUTPUT_DIR}')
print(f'  - Métadonnées: {"OK" if METADATA_DIR.exists() else "ERREUR"}')
print(f'  - Images: {"OK" if IMG_DIR.exists() else "ERREUR"}')

## Cellule 3: Chargement des Attributs (40 attributs)

In [ ]:
# Définir les 40 attributs CelebA
ATTRIBUTE_NAMES = [
    '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes',
    'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair',
    'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin',
    'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones',
    'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard',
    'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks',
    'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings',
    'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'
]

print(f'✓ Chargement de {len(ATTRIBUTE_NAMES)} attributs')

# Charger les attributs
attr_file = METADATA_DIR / 'list_attr_celeba.txt'
attributes = {}

with open(attr_file, 'r') as f:
    lines = f.readlines()

# Ignorer les 2 premières lignes (count et noms d'attributs)
for line in lines[2:]:
    parts = line.strip().split()
    if len(parts) >= 41:
        img_name = parts[0]
        attr_values = [int(v) for v in parts[1:41]]
        # Convertir -1/1 à 0/1
        attr_dict = {attr_name: 1 if val > 0 else 0 
                    for attr_name, val in zip(ATTRIBUTE_NAMES, attr_values)}
        attributes[img_name] = attr_dict

print(f'✓ Attributs chargés pour {len(attributes)} images')
print(f'\nExemple d\'attributs pour la première image:')
first_img = list(attributes.keys())[0]
print(f'  Image: {first_img}')
print(f'  Genre (Male): {attributes[first_img]["Male"]}')
print(f'  Jeune (Young): {attributes[first_img]["Young"]}')
print(f'  Sourire (Smiling): {attributes[first_img]["Smiling"]}')

## Cellule 4: Chargement des Identités

In [ ]:
# Charger les identités
id_file = METADATA_DIR / 'identity_CelebA.txt'
identities = {}

with open(id_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            img_name, person_id = parts
            identities[img_name] = int(person_id)

print(f'✓ Identités chargées pour {len(identities)} images')
print(f'\nStatistiques d\'identité:')
print(f'  Personnes uniques: {len(set(identities.values()))}')
print(f'  ID personne max: {max(identities.values())}')
print(f'  ID personne min: {min(identities.values())}')

# Compter les images par personne
images_per_person = Counter(identities.values())
print(f'\nImages par personne:')
print(f'  Moyenne: {np.mean(list(images_per_person.values())):.1f}')
print(f'  Maximum: {max(images_per_person.values())}')
print(f'  Minimum: {min(images_per_person.values())}')

## Cellule 5: Création du DataFrame Principal

In [ ]:
# Créer le dataframe
data = []
for img_name in sorted(attributes.keys()):
    row = {'image_id': img_name}
    row.update(attributes[img_name])
    if img_name in identities:
        row['identity_id'] = identities[img_name]
    data.append(row)

df = pd.DataFrame(data)
print(f'✓ DataFrame créé avec shape {df.shape}')
print(f'\nInformations du DataFrame:')
print(f'  Lignes totales: {len(df)}')
print(f'  Colonnes totales: {len(df.columns)}')
print(f'\nPremières lignes:')
df.head(3)

## Cellule 6: Analyse de la Distribution des Attributs

In [ ]:
# Calculer les statistiques d'attribut
attr_stats = {}
for attr in ATTRIBUTE_NAMES:
    count = (df[attr] == 1).sum()
    percentage = count / len(df) * 100
    attr_stats[attr] = {'count': count, 'percentage': percentage}

# Trier par pourcentage
attr_df = pd.DataFrame(attr_stats).T.sort_values('percentage', ascending=False)

print('Top 15 Attributs les Plus Communs:')
print(attr_df.head(15))

print('\nTop 15 Attributs les Moins Communs:')
print(attr_df.tail(15))

## Cellule 7: Graphique 1 - Distribution de Tous les Attributs

In [ ]:
# Graphique de tous les attributs
fig, ax = plt.subplots(figsize=(16, 8))

attrs = attr_df.index.tolist()
percentages = attr_df['percentage'].values

colors = plt.cm.viridis(np.linspace(0, 1, len(attrs)))
bars = ax.barh(attrs, percentages, color=colors)

ax.set_xlabel('Pourcentage d\'images (%)', fontsize=12, fontweight='bold')
ax.set_title('CelebA: Distribution des 40 Attributs', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Ajouter les étiquettes de pourcentage
for i, (attr, pct) in enumerate(zip(attrs, percentages)):
    ax.text(pct + 1, i, f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_tous_les_attributs.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "01_tous_les_attributs.png"}')

## Cellule 8: Graphique 2 - Distribution du Genre

In [ ]:
# Distribution du genre
male_count = (df['Male'] == 1).sum()
female_count = (df['Male'] == 0).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique en camembert
labels = ['Hommes', 'Femmes']
sizes = [male_count, female_count]
colors = ['#FF6B6B', '#4ECDC4']
explode = (0.05, 0.05)

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax1.set_title('Distribution du Genre', fontsize=13, fontweight='bold')

# Graphique en barres
ax2.bar(labels, sizes, color=colors, edgecolor='black', linewidth=2)
ax2.set_ylabel('Nombre d\'images', fontsize=11, fontweight='bold')
ax2.set_title('Nombre d\'images par Genre', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for i, v in enumerate(sizes):
    ax2.text(i, v + 2000, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_distribution_genre.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Hommes: {male_count} ({male_count/len(df)*100:.1f}%)')
print(f'Femmes: {female_count} ({female_count/len(df)*100:.1f}%)')
print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "02_distribution_genre.png"}')

## Cellule 9: Graphique 3 - Distribution de l'Âge

In [ ]:
# Distribution de l'âge
young_count = (df['Young'] == 1).sum()
old_count = (df['Young'] == 0).sum()

fig, ax = plt.subplots(figsize=(8, 6))

labels = ['Jeunes', 'Plus âgés']
sizes = [young_count, old_count]
colors = ['#FFD93D', '#6BCB77']
explode = (0.05, 0.05)

ax.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
       shadow=True, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax.set_title('Distribution de l\'Âge (Jeunes vs Plus Âgés)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_distribution_age.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Jeunes: {young_count} ({young_count/len(df)*100:.1f}%)')
print(f'Plus âgés: {old_count} ({old_count/len(df)*100:.1f}%)')
print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "03_distribution_age.png"}')

## Cellule 10: Graphique 4 - Distribution de l'Expression

In [ ]:
# Distribution de l'expression
smiling = (df['Smiling'] == 1).sum()
mouth_open = (df['Mouth_Slightly_Open'] == 1).sum()
serious = len(df) - smiling - mouth_open

fig, ax = plt.subplots(figsize=(10, 6))

expressions = ['Sourire', 'Bouche ouverte', 'Sérieux/Autre']
counts = [smiling, mouth_open, serious]
colors = ['#FF6B6B', '#FFA06B', '#6BCB77']

bars = ax.bar(expressions, counts, color=colors, edgecolor='black', linewidth=2)
ax.set_ylabel('Nombre d\'images', fontsize=11, fontweight='bold')
ax.set_title('Distribution de l\'Expression Faciale', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{count}\n({count/len(df)*100:.1f}%)',
           ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_distribution_expression.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "04_distribution_expression.png"}')

## Cellule 11: Graphique 5 - Distribution des Couleurs de Cheveux

In [ ]:
# Distribution des couleurs de cheveux
hair_colors = {
    'Cheveux Noirs': (df['Black_Hair'] == 1).sum(),
    'Cheveux Bruns': (df['Brown_Hair'] == 1).sum(),
    'Cheveux Blonds': (df['Blond_Hair'] == 1).sum(),
    'Cheveux Gris': (df['Gray_Hair'] == 1).sum(),
    'Chauve': (df['Bald'] == 1).sum(),
}

# Trier par nombre
hair_colors = dict(sorted(hair_colors.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 6))

colors_list = ['#000000', '#8B4513', '#FFD700', '#808080', '#C0C0C0']
bars = ax.bar(hair_colors.keys(), hair_colors.values(), color=colors_list, 
             edgecolor='black', linewidth=2)
ax.set_ylabel('Nombre d\'images', fontsize=11, fontweight='bold')
ax.set_title('Distribution des Couleurs de Cheveux', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)

for bar, count in zip(bars, hair_colors.values()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{count}\n({count/len(df)*100:.1f}%)',
           ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_distribution_couleurs_cheveux.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "05_distribution_couleurs_cheveux.png"}')

## Cellule 12: Graphique 6 - Distribution de la Pilosité Faciale (Hommes)

In [ ]:
# Pilosité faciale (hommes uniquement)
male_df = df[df['Male'] == 1]

facial_hair = {
    'Ombre 5 o\'clock': (male_df['5_o_Clock_Shadow'] == 1).sum(),
    'Moustache': (male_df['Mustache'] == 1).sum(),
    'Bouc': (male_df['Goatee'] == 1).sum(),
    'Favoris': (male_df['Sideburns'] == 1).sum(),
    'Sans barbe': (male_df['No_Beard'] == 1).sum(),
}

facial_hair = dict(sorted(facial_hair.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(facial_hair.keys(), facial_hair.values(), 
             color='#8B7355', edgecolor='black', linewidth=2)
ax.set_ylabel('Nombre d\'images (hommes)', fontsize=11, fontweight='bold')
ax.set_title('Distribution de la Pilosité Faciale (Hommes uniquement)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, facial_hair.values()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{count}\n({count/len(male_df)*100:.1f}%)',
           ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_pilosite_faciale.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total hommes: {len(male_df)}')
print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "06_pilosite_faciale.png"}')

## Cellule 13: Graphique 7 - Distribution des Accessoires

In [ ]:
# Distribution des accessoires
accessories = {
    'Chapeau': (df['Wearing_Hat'] == 1).sum(),
    'Lunettes': (df['Eyeglasses'] == 1).sum(),
    'Boucles d\'oreilles': (df['Wearing_Earrings'] == 1).sum(),
    'Collier': (df['Wearing_Necklace'] == 1).sum(),
    'Cravate': (df['Wearing_Necktie'] == 1).sum(),
    'Rouge à lèvres': (df['Wearing_Lipstick'] == 1).sum(),
}

accessories = dict(sorted(accessories.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.bar(accessories.keys(), accessories.values(), 
             color='#FF69B4', edgecolor='black', linewidth=2)
ax.set_ylabel('Nombre d\'images', fontsize=11, fontweight='bold')
ax.set_title('Distribution des Accessoires', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, accessories.values()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{count}\n({count/len(df)*100:.1f}%)',
           ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_distribution_accessoires.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "07_distribution_accessoires.png"}')

## Cellule 14: Graphique 8 - Distribution des Identités par Images

In [ ]:
# Distribution des identités
images_per_person = df['identity_id'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(images_per_person.values, bins=50, color='#4ECDC4', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Nombre d\'images par personne', fontsize=11, fontweight='bold')
ax.set_ylabel('Nombre de personnes', fontsize=11, fontweight='bold')
ax.set_title('Distribution des Images par Identité', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Ajouter les statistiques
stats_text = f'Moyenne: {images_per_person.mean():.1f}\n' \
            f'Médiane: {images_per_person.median():.0f}\n' \
            f'Max: {images_per_person.max()}\n' \
            f'Min: {images_per_person.min()}'
ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
       fontsize=11, verticalalignment='top', horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
       family='monospace', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_distribution_identites.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "08_distribution_identites.png"}')

## Cellule 15: Graphique 9 - Matrice de Corrélation des Attributs

In [ ]:
# Sélectionner les attributs clés pour la corrélation
key_attrs = [
    'Male', 'Young', 'Smiling', 'Eyeglasses', 'Wearing_Hat',
    'Black_Hair', 'Brown_Hair', 'Blond_Hair', 'High_Cheekbones',
    'Heavy_Makeup', 'Mustache', 'Goatee', 'Bald', 'Attractive'
]

corr_matrix = df[key_attrs].corr()

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
           center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
           ax=ax, vmin=-0.5, vmax=0.5)

ax.set_title('Matrice de Corrélation des Attributs Clés', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '09_matrice_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "09_matrice_correlation.png"}')

## Cellule 16: Graphique 10 - Combinaisons d'Attributs les Plus Fréquentes

In [ ]:
# Combinaisons d'attributs les plus courantes
combinations = []

for idx, row in df.iterrows():
    attrs = []
    if row['Male'] == 1:
        attrs.append('Homme')
    else:
        attrs.append('Femme')
    
    if row['Young'] == 1:
        attrs.append('Jeune')
    else:
        attrs.append('Âgé')
    
    if row['Smiling'] == 1:
        attrs.append('Souriant')
    else:
        attrs.append('Sérieux')
    
    combo = ' + '.join(attrs)
    combinations.append(combo)

df['combination'] = combinations
combo_counts = df['combination'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(range(len(combo_counts)), combo_counts.values, color='#FF6B6B')
ax.set_yticks(range(len(combo_counts)))
ax.set_yticklabels(combo_counts.index)
ax.set_xlabel('Nombre d\'images', fontsize=11, fontweight='bold')
ax.set_title('Top 10 Combinaisons d\'Attributs', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(combo_counts.values):
    ax.text(v + 500, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_top_combinaisons.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Graphique sauvegardé: {OUTPUT_DIR / "10_top_combinaisons.png"}')

## Cellule 17: Résumé Statistique Complet

In [ ]:
# Résumé complet
summary = {
    'Images Totales': len(df),
    'Personnes Uniques': int(df['identity_id'].nunique()),
    'Attributs Totaux': len(ATTRIBUTE_NAMES),
    '---': '---',
    'Hommes': (df['Male'] == 1).sum(),
    'Femmes': (df['Male'] == 0).sum(),
    'Jeunes': (df['Young'] == 1).sum(),
    'Âgés': (df['Young'] == 0).sum(),
    '---2': '---',
    'Souriants': (df['Smiling'] == 1).sum(),
    'Portant Lunettes': (df['Eyeglasses'] == 1).sum(),
    'Portant Chapeau': (df['Wearing_Hat'] == 1).sum(),
    'Chauves': (df['Bald'] == 1).sum(),
}

print('\n' + '='*50)
print('RÉSUMÉ DU DATASET CELEBA')
print('='*50)
for key, value in summary.items():
    if key.startswith('---'):
        print('')
    else:
        print(f'{key:.<30} {value:>10}')
print('='*50)

## Cellule 18: Exportation des Résultats d'Analyse

In [ ]:
import json

# Sauvegarder les résultats d'analyse
analysis_results = {
    'dataset_size': len(df),
    'unique_persons': int(df['identity_id'].nunique()),
    'total_attributes': len(ATTRIBUTE_NAMES),
    'attribute_statistics': attr_df[['count', 'percentage']].to_dict(),
    'gender_distribution': {
        'male': int((df['Male'] == 1).sum()),
        'female': int((df['Male'] == 0).sum())
    },
    'age_distribution': {
        'young': int((df['Young'] == 1).sum()),
        'older': int((df['Young'] == 0).sum())
    },
    'images_per_person': {
        'mean': float(images_per_person.mean()),
        'median': float(images_per_person.median()),
        'max': int(images_per_person.max()),
        'min': int(images_per_person.min())
    }
}

with open(OUTPUT_DIR / 'celeba_analysis.json', 'w') as f:
    json.dump(analysis_results, f, indent=2)

print(f'✓ Analyse sauvegardée: {OUTPUT_DIR / "celeba_analysis.json"}')

# Sauvegarder les statistiques d'attributs
attr_df.to_csv(OUTPUT_DIR / 'attribute_statistics.csv')
print(f'✓ Statistiques d\'attributs sauvegardées: {OUTPUT_DIR / "attribute_statistics.csv"}')

print('\n🎉 EXPLORATION TERMINÉE!')
print(f'   - {len(df)} images analysées')
print(f'   - 10 graphiques générés')
print(f'   - 2 fichiers de résultats créés')